# Feldman-Cousins confidence intervals

## Motivation: the NOMAD problem and empty intervals

This notebook presents the **Feldman-Cousins** method ([Phys. Rev. D 57, 3873, 1998](https://arxiv.org/abs/physics/9711021))
for constructing unified confidence intervals in Poisson counting experiments.

**Contents:**
1. The NOMAD problem: expecting background and seeing nothing
2. Classical Neyman intervals and their pathologies
3. The Feldman-Cousins ordering principle
4. Construction of the confidence belt
5. Numerical examples

## Setup

In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os, sys
from pathlib import Path

_env = os.environ.get('FANAL_ROOT') or os.environ.get('USCFANALDIR')
if _env and Path(_env, 'data').is_dir():
    rootpath = str(Path(_env).resolve())
else:
    rootpath = str(next(p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'data').is_dir() and (p / 'ana').is_dir()))
if rootpath not in sys.path:
    sys.path.insert(0, rootpath)

import core.pltext as pltext
pltext.style()

import ana.crib_fc as crib_fc

print('Fanal root:', rootpath)

---
## 1. The NOMAD problem

The **NOMAD** experiment (CERN, search for $\nu_\mu \to \nu_\tau$ oscillations) expected to observe
$b = 3$ background events in its signal region.

**Question:** What happens if we observe $n = 0$ events?

With $b = 3$ expected background events, the probability of observing zero events is:

$$P(n=0 \mid \mu + b) = e^{-(\mu + b)}$$

For $\mu = 0$ (no signal): $P(0 \mid 3) = e^{-3} \approx 0.05$.

Unlikely, but not impossible. This scenario — expecting background and seeing nothing —
creates a serious problem with classical confidence intervals.

In [ ]:
crib_fc.plot_nomad_poisson(bkg=3.0)

---
## 2. The problem with classical intervals

### Neyman construction for Poisson

For a Poisson process with mean $\mu + b$ (signal + background), the classical
**central** confidence interval at level $\alpha$ is constructed as follows:

For each true value of $\mu$, we find the acceptance interval $[n_1, n_2]$ such that:

$$P(n_1 \leq n \leq n_2 \mid \mu + b) \geq \alpha$$

splitting the remaining probability **equally** between both tails:

$$P(n < n_1 \mid \mu + b) \leq \frac{1-\alpha}{2}, \quad P(n > n_2 \mid \mu + b) \leq \frac{1-\alpha}{2}$$

The confidence interval for an observed $n$ is obtained by inverting this band:
the set of $\mu$ values whose acceptance intervals contain $n_{obs}$.

### The problem: empty intervals

With the central ordering, if $b = 3$ and we observe $n = 0$:

- The maximum likelihood estimator is $\hat{\mu} = n - b = 0 - 3 = -3$ (unphysical!)
- The central interval at 90% CL turns out to be **empty**: there is no $\mu \geq 0$
  whose acceptance interval includes $n = 0$

This is absurd: we observed real data and cannot say anything about $\mu$.

In [ ]:
crib_fc.plot_classical_band(bkg=3.0, cl=0.90, n_obs=0)

### The flip-flopping problem

Beyond empty intervals, classical intervals suffer from **flip-flopping**:

- If the experimenter observes $n$ large (relative to $b$), they report a **central** two-sided interval for $\mu$.
- If they observe $n$ small (compatible with or below $b$), they switch to a one-sided **upper limit**.

This *a posteriori* decision (based on the data) **violates frequentist coverage**.
The 90% confidence level no longer means the interval contains the true value
90% of the time, because the construction rule changes depending on the outcome.

| Scenario | Classical decision | Problem |
|-----------|-----------------|----------|
| $n \gg b$ | Two-sided interval | OK |
| $n \approx b$ | Two-sided or upper limit? | Ambiguity |
| $n < b$ | Upper limit | Under-coverage |
| $n = 0, b > 0$ | Upper limit... or empty? | Empty interval |

**Feldman and Cousins solve both problems** with a single procedure that automatically
transitions between upper limits and two-sided intervals.

---
## 3. The Feldman-Cousins ordering method

### Key idea: ordering by likelihood ratio

The Neyman construction requires choosing **which values of $n$ to include first** in the
acceptance interval of each $\mu$. The classical central ordering splits the tails equally.
Feldman-Cousins proposes a different criterion: **ordering by the likelihood ratio**.

For each possible observation $n$, compute:

$$R(n) = \frac{P(n \mid \mu)}{P(n \mid \hat{\mu}_{\text{best}})}$$

where:
- $P(n \mid \mu) = \frac{(\mu + b)^n \, e^{-(\mu+b)}}{n!}$ is the Poisson probability of observing $n$ given $\mu$ and $b$
- $\hat{\mu}_{\text{best}} = \max(n - b, \, 0)$ is the **best physical estimator** of $\mu$ given $n$
  (constrained to $\mu \geq 0$, since the signal cannot be negative)
- $P(n \mid \hat{\mu}_{\text{best}})$ is the probability evaluated at the best fit

Equivalently, we use the test statistic:

$$t(n) = -2 \ln R(n) = -2 \left[\ln P(n \mid \mu) - \ln P(n \mid \hat{\mu}_{\text{best}})\right]$$

**Values of $n$ with the smallest $t$ (largest $R$) are included first.**

### Step-by-step algorithm

For a fixed value of $\mu$ and background $b$, at confidence level CL:

1. For each $n = 0, 1, 2, \ldots, n_{\max}$:
   - Compute $\hat{\mu}_{\text{best}}(n) = \max(n - b, 0)$
   - Compute $R(n) = P(n \mid \mu + b) \,/\, P(n \mid \hat{\mu}_{\text{best}} + b)$
   
2. Sort the values of $n$ from largest to smallest $R$ (equivalently, from smallest to largest $t$)

3. Include values of $n$ in that order, accumulating $P(n \mid \mu + b)$, until the cumulative probability reaches CL

4. The acceptance interval is $[n_{\min}, n_{\max}]$ of the included values

5. Repeat for each $\mu$ on a fine grid

6. **Invert**: for a given $n_{\text{obs}}$, the confidence interval is the set of $\mu$ values whose acceptance intervals contain $n_{\text{obs}}$

### Worked example: FC ordering for $\mu = 1$, $b = 3$

Let us walk through step by step how FC selects which values of $n$ to include in the acceptance interval.

In [ ]:
crib_fc.print_fc_ordering_table(mu=1.0, bkg=3.0)

In [ ]:
crib_fc.print_fc_inclusion_order(mu=1.0, bkg=3.0, cl=0.90)

**Key observation:** Note that for $n < b$ (e.g. $n = 0, 1, 2$), the estimator
$\hat\mu_\text{best} = 0$ (it saturates at the physical boundary), and the ratio $R$ is not
necessarily small. This allows $n=0$ **to be included** in the acceptance interval of small
values of $\mu$, resolving the empty interval problem.

---
## 4. Building the full FC confidence belt

In [ ]:
crib_fc.plot_fc_belt(bkg=3.0, cl=0.90)

### Comparison: classical central vs Feldman-Cousins

Let us overlay both bands to understand the difference.

In [ ]:
crib_fc.plot_comparison_classical_fc(bkg=3.0, cl=0.90)

---
## 5. Automatic transition: upper limit ↔ two-sided interval

One of the main virtues of FC is that it **does not require deciding a priori** whether to report
an upper limit or a two-sided interval. The transition occurs naturally:

- For **small** $n_\text{obs}$ (less than or close to $b$): the FC interval is asymmetric
  and starts at $\mu = 0$ -- it acts as an **upper limit**
- For **large** $n_\text{obs}$ (well above $b$): the FC interval centres around
  $\hat\mu = n - b$ -- it acts as a **two-sided interval**

This solves the *flip-flopping* problem: the same procedure, applied blindly before
seeing the data, produces the correct type of interval.

In [ ]:
crib_fc.plot_fc_transition(bkg=3.0, cl=0.90)

---
## 6. Effect of background on the FC belt

How does the confidence belt change with the background level? Let us compare $b = 0$, $b = 3$, and $b = 10$.

In [ ]:
crib_fc.plot_fc_vs_background(bkgs=(0., 3., 10.), cl=0.90)

---
## 7. Comparison of confidence levels

Let us see how the FC intervals change at 68%, 90%, and 95% CL for the $b = 3$ case.

In [ ]:
crib_fc.plot_fc_vs_cl(bkg=3.0, cls=(0.95, 0.90, 0.68))

---
## Summary

| Aspect | Classical central interval | Feldman-Cousins |
|---------|--------------------------|-----------------|
| Ordering | Equal tails $(1-\alpha)/2$ | Likelihood ratio $R(n)$ |
| $n \ll b$ | Empty interval | Upper limit (valid) |
| $n \gg b$ | Two-sided interval | Two-sided interval |
| Prior decision | Must choose UL vs two-sided | Automatic |
| Coverage | Violated by *flip-flopping* | Guaranteed by construction |
| Physical constraint $\mu \geq 0$ | Not incorporated | Built in via $\hat\mu_{\text{best}}$ |

**Reference:** G.J. Feldman and R.D. Cousins, *"Unified approach to the classical statistical
analysis of small signals"*, Phys. Rev. D 57, 3873 (1998). [arXiv:physics/9711021](https://arxiv.org/abs/physics/9711021)